In [91]:
import pandas as pd

In [92]:
dfAnimeWorks = pd.read_csv("../data/raw/character_anime_works.csv")
dfNicknames = pd.read_csv("../data/raw/character_nicknames.csv")
dfCharacters = pd.read_csv("../data/raw/characters.csv")

# Cleaning the data
### Characters df
##### Removing columns that are not needed

In [93]:
dfCharacters = dfCharacters.drop(columns=["url",
                                          "name_kanji",
                                          "image",
                                          "about"],
                                 errors="ignore")

### Normalizing
##### Favorites cast to int and set NaN to 0

In [94]:
dfCharacters.drop_duplicates()
dfCharacters["favorites"] = (
    pd.to_numeric(dfCharacters["favorites"], errors="coerce")
      .fillna(0)
      .astype(int)
)

##### ID set to int and removed Nan values

In [95]:
dfCharacters = dfCharacters[dfCharacters["character_mal_id"] != 0]
dfCharacters["character_mal_id"] = (
    pd.to_numeric(dfCharacters["character_mal_id"], errors="coerce")
        .fillna(-1)
        .astype(int)
)
dfCharacters = dfCharacters[dfCharacters["character_mal_id"] != -1]

### Nicknames df
##### Removed rows where nickname was empty or Nan, useless

In [96]:
dfNicknames = dfNicknames[dfNicknames["nickname"].notna()]

##### Some Characters have multiple nicknames, so it's better to put them in the same row as a list

In [97]:
dfNicknames = (
    dfNicknames
    .groupby("character_mal_id")["nickname"]
    .apply(list)
    .reset_index()
)


In [98]:
dfNicknames

,character_mal_id,nickname
0,3,"[Running Rock, Black Dog]"
1,5,"[Ichi-nii, Shinigami Daiko (Substitute Soul Re..."
2,7,[Hime]
3,11,"[Ed, Fullmetal Alchemist, Hagane no shounen, C..."
4,12,"[Al, Armored Alchemist]"
...,...,...
30262,282048,[Great Zuma]
30263,282159,"[Ling Long, Silvermoon]"
30264,282227,[Mei's Mother]
30265,282254,[Cyrano]


### AnimeWorks df

In [99]:
dfAnimeWorks

,anime_mal_id,character_mal_id,character_name,role
0,2928,5781,Atoli,Main
1,2928,33,Haseo,Main
2,2928,32,Ovan,Main
3,2928,34,Shino,Main
4,2928,5785,Aina,Supporting
...,...,...,...,...
236811,31245,137157,"Shibasaki, Ken",Supporting
236812,36305,136064,"Hamanaka, Midori",Main
236813,36305,133916,"Narumi, Sena",Main
236814,36305,124942,"Hayasaka, Akari",Supporting


# Merging

In [100]:
dfCharacters = dfCharacters.merge(dfNicknames, on="character_mal_id", how = "left")
dfNicknames = None

##### have to set the Nan values as empty lists

In [101]:
dfCharacters['nickname'] = dfCharacters['nickname'].apply(lambda x: x if isinstance(x, list) else [])

In [ ]:
dfCharacters.to_csv("../data/cleaned/characters.csv", index=False)
dfAnimeWorks.to_csv("../data/cleaned/anime_works.csv", index=False)

In [104]:
dfCharacters = None
dfAnimeWorks = None